# ONNX Semantic Security Engine -- Full 15-Class Training

**Dataset:** CSE-CIC-IDS2018 (all 10 days, parquet format)  
**Model:** ThreatMLP -- 256 -> 128 -> 64 -> 15 classes with BatchNorm  
**Pipeline:**
1. Load all 10 parquet files, drop leakage cols, resample
2. Train MLP with LR scheduler + best-model checkpointing (30 epochs)
3. Export FP32 ONNX, Validate PyTorch == ONNX (RQ1)
4. INT8 static quantization, size/latency/F1 comparison (RQ2)
5. Cross-dataset evaluation on ToN-IoT (RQ3)

**Target:** Macro-F1 >= 0.80 across all 15 classes  
**Download after run:** threat_mlp.pth, label_encoder.joblib, standard_scaler.joblib, threat_mlp_fp32.onnx, threat_mlp_int8.onnx

In [ ]:
!pip install -q onnx onnxscript onnxruntime


In [ ]:
import torch
import torch.nn as nn

# Widened architecture: 256 -> 128 -> 64 -> num_classes
# BatchNorm added after each hidden layer for stable 15-class training
class ThreatMLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(ThreatMLP, self).__init__()
        self.fc1      = nn.Linear(input_dim, 256)
        self.bn1      = nn.BatchNorm1d(256)
        self.relu1    = nn.ReLU()
        self.dropout1 = nn.Dropout(0.3)
        self.fc2      = nn.Linear(256, 128)
        self.bn2      = nn.BatchNorm1d(128)
        self.relu2    = nn.ReLU()
        self.dropout2 = nn.Dropout(0.3)
        self.fc3      = nn.Linear(128, 64)
        self.bn3      = nn.BatchNorm1d(64)
        self.relu3    = nn.ReLU()
        self.dropout3 = nn.Dropout(0.3)
        self.fc4      = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.dropout1(self.relu1(self.bn1(self.fc1(x))))
        x = self.dropout2(self.relu2(self.bn2(self.fc2(x))))
        x = self.dropout3(self.relu3(self.bn3(self.fc3(x))))
        return self.fc4(x)

print("ThreatMLP defined (256->128->64->num_classes with BatchNorm)")


In [ ]:
import pandas as pd
import numpy as np
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.utils.class_weight import compute_class_weight
import joblib, os, time, glob

# Load all parquet files
parquet_files = glob.glob("/kaggle/input/datasets/dhoogla/csecicids2018/*.parquet", recursive=True)
print(f"Found {len(parquet_files)} parquet files.")
dfs = []
for file in parquet_files:
    print(f"Loading {os.path.basename(file)}...")
    dfs.append(pd.read_parquet(file))
df = pd.concat(dfs, ignore_index=True)
df.columns = df.columns.str.strip()

# Drop leakage columns (IP/port/protocol are dataset-specific identifiers,
# not semantic features -- per Cantone et al. 2024, Paper 6 in lit review)
leakage_cols = ["Timestamp", "Flow ID", "Src IP", "Dst IP", "Src Port", "Dst Port", "Protocol"]
df = df.drop(columns=[c for c in leakage_cols if c in df.columns], errors="ignore")
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df = df.dropna().drop_duplicates()
print(f"Raw dataset shape: {df.shape}")
print(df['Label'].value_counts().to_string())

# Resample: cap majority at 100K, oversample minority to 5K
MAX_SAMPLES = 100_000
MIN_SAMPLES = 5_000
resampled_dfs = []
for label, group in df.groupby("Label"):
    n = len(group)
    if n > MAX_SAMPLES:
        resampled_dfs.append(group.sample(n=MAX_SAMPLES, random_state=42))
        print(f"  down {label}: {n} -> {MAX_SAMPLES}")
    elif n < MIN_SAMPLES:
        resampled_dfs.append(group.sample(n=MIN_SAMPLES, replace=True, random_state=42))
        print(f"  up   {label}: {n} -> {MIN_SAMPLES}")
    else:
        resampled_dfs.append(group)
        print(f"  =    {label}: {n}")
df = pd.concat(resampled_dfs, ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Resampled dataset shape: {df.shape}")


In [ ]:
# Encode, split, scale, train
X = df.drop(columns=['Label']).values
y = df['Label'].values
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
num_classes = len(label_encoder.classes_)
print(f"Classes ({num_classes}): {label_encoder.classes_}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

# Scale AFTER split -- fit only on train to prevent data leakage
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)
np.save("/kaggle/working/X_test.npy", X_test)
np.save("/kaggle/working/y_test.npy", y_test)

X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.LongTensor(y_train)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

input_dim = X_train.shape[1]
model = ThreatMLP(input_dim, num_classes).to(device)

class_weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
criterion = nn.CrossEntropyLoss(weight=torch.FloatTensor(class_weights).to(device))
optimizer = optim.Adam(model.parameters(), lr=1e-3)
# ReduceLROnPlateau: halve LR when loss stops improving for 3 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=512, shuffle=True)

NUM_EPOCHS = 30
best_f1 = 0.0
model.train()
for epoch in range(NUM_EPOCHS):
    total_loss, n = 0.0, 0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        loss = criterion(model(bx), by)
        loss.backward()
        optimizer.step()
        total_loss += loss.item(); n += 1
    avg_loss = total_loss / n
    scheduler.step(avg_loss)
    # Evaluate every 5 epochs on test set
    if (epoch + 1) % 5 == 0:
        model.eval()
        with torch.no_grad():
            val_preds = torch.argmax(
                model(torch.FloatTensor(X_test).to(device)), dim=1
            ).cpu().numpy()
            val_f1 = f1_score(y_test, val_preds, average="macro")
        model.train()
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), "/kaggle/working/threat_mlp_best.pth")
        print(f"Epoch {epoch+1}/{NUM_EPOCHS} -- Loss: {avg_loss:.4f} -- Val F1: {val_f1:.4f} -- Best: {best_f1:.4f}")
    else:
        print(f"Epoch {epoch+1}/{NUM_EPOCHS} -- Loss: {avg_loss:.4f}")

# Load best checkpoint and final evaluation
model.load_state_dict(torch.load("/kaggle/working/threat_mlp_best.pth"))
model.eval()
with torch.no_grad():
    preds = torch.argmax(model(torch.FloatTensor(X_test).to(device)), dim=1).cpu().numpy()
    fp32_f1 = f1_score(y_test, preds, average="macro")
    print(f"Best Model -- Macro-F1: {fp32_f1:.4f}")
    print(classification_report(y_test, preds, target_names=label_encoder.classes_))

# Save artifacts (download these from Kaggle output after the run)
torch.save(model.state_dict(), "/kaggle/working/threat_mlp.pth")
joblib.dump(label_encoder, "/kaggle/working/label_encoder.joblib")
joblib.dump(scaler, "/kaggle/working/standard_scaler.joblib")
print("Model + scaler + encoder saved")


In [ ]:
import onnx
import onnxruntime as ort

# Export FP32 ONNX (move to CPU first)
model.cpu().eval()
dummy = torch.randn(1, input_dim, dtype=torch.float32)
torch.onnx.export(
    model, dummy,
    "/kaggle/working/threat_mlp_fp32.onnx",
    opset_version=17,
    input_names=["input"], output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
)
onnx_model = onnx.load("/kaggle/working/threat_mlp_fp32.onnx")
onnx.checker.check_model(onnx_model)
print("FP32 ONNX structure valid")

# RQ1 Semantic Validation: compare PyTorch vs ONNX Runtime predictions
# Per Jajal et al. 2024 (Paper 2), 33% of ONNX conversions fail silently.
ort_session = ort.InferenceSession("/kaggle/working/threat_mlp_fp32.onnx", providers=["CPUExecutionProvider"])
all_match = True
for i in range(20):
    rand_input = np.random.randn(1, input_dim).astype(np.float32)
    with torch.no_grad():
        pt_pred = torch.argmax(model(torch.FloatTensor(rand_input)), dim=1).item()
    onnx_pred = int(np.argmax(ort_session.run(None, {"input": rand_input})[0], axis=1)[0])
    status = "MATCH" if pt_pred == onnx_pred else "MISMATCH"
    if pt_pred != onnx_pred: all_match = False
    print(f"Sample {i+1:2d}: PyTorch={pt_pred}  ONNX={onnx_pred}  {status}")
print("RQ1 PASSED: All samples match" if all_match else "RQ1 FAILED: Mismatch detected")


In [ ]:
from onnxruntime.quantization import quantize_static, QuantType, QuantFormat, CalibrationDataReader

class CICCalibrationReader(CalibrationDataReader):
    def __init__(self, X_cal):
        self.data = [X_cal[i:i+1].astype(np.float32) for i in range(len(X_cal))]
        self.idx = 0
    def get_next(self):
        if self.idx >= len(self.data): return None
        result = {"input": self.data[self.idx]}
        self.idx += 1
        return result
    def rewind(self): self.idx = 0

cal_reader = CICCalibrationReader(X_train[:1000])
quantize_static(
    model_input="/kaggle/working/threat_mlp_fp32.onnx",
    model_output="/kaggle/working/threat_mlp_int8.onnx",
    calibration_data_reader=cal_reader,
    quant_format=QuantFormat.QDQ,
    weight_type=QuantType.QInt8,
    activation_type=QuantType.QUInt8,
)
print("INT8 model saved")


In [ ]:
import json

def benchmark_model(model_path, X_data, y_data, label, n_runs=1000):
    sess = ort.InferenceSession(model_path, providers=["CPUExecutionProvider"])
    total_size = os.path.getsize(model_path)
    data_file = model_path + '.data'
    if os.path.exists(data_file): total_size += os.path.getsize(data_file)
    size_mb = total_size / (1024 * 1024)
    sample = X_data[:1].astype(np.float32)
    for _ in range(10):
        sess.run(None, {"input": sample})
    start = time.perf_counter()
    for i in range(n_runs):
        sess.run(None, {"input": X_data[i:i+1].astype(np.float32)})
    elapsed_ms = (time.perf_counter() - start) / n_runs * 1000
    all_preds = []
    for i in range(0, len(X_data), 512):
        batch = X_data[i:i+512].astype(np.float32)
        all_preds.extend(np.argmax(sess.run(None, {"input": batch})[0], axis=1))
    macro_f1 = f1_score(y_data, all_preds, average="macro")
    print(f"\n{label}  |  Size: {size_mb:.3f} MB  |  Latency: {elapsed_ms:.3f} ms/sample  |  F1: {macro_f1:.4f}")
    return {"model": label, "size_mb": size_mb, "latency_ms": elapsed_ms, "macro_f1": macro_f1}

X_test_bm = np.load("/kaggle/working/X_test.npy")
y_test_bm = np.load("/kaggle/working/y_test.npy")
fp32_results = benchmark_model("/kaggle/working/threat_mlp_fp32.onnx", X_test_bm, y_test_bm, "FP32")
int8_results = benchmark_model("/kaggle/working/threat_mlp_int8.onnx", X_test_bm, y_test_bm, "INT8")

size_pct = (1 - int8_results['size_mb'] / fp32_results['size_mb']) * 100
lat_pct  = (1 - int8_results['latency_ms'] / fp32_results['latency_ms']) * 100
f1_delta = int8_results['macro_f1'] - fp32_results['macro_f1']
print("\nRQ2 SUMMARY")
print(f"Size:    {fp32_results['size_mb']:.3f} -> {int8_results['size_mb']:.3f} MB  ({size_pct:.1f}% smaller)")
print(f"Latency: {fp32_results['latency_ms']:.3f} -> {int8_results['latency_ms']:.3f} ms")
print(f"F1:      {fp32_results['macro_f1']:.4f} -> {int8_results['macro_f1']:.4f}  (delta={f1_delta:+.4f})")

with open("/kaggle/working/quantization_comparison.json", "w") as f:
    json.dump({"fp32": fp32_results, "int8": int8_results}, f, indent=2)
print("Results saved to quantization_comparison.json")


In [ ]:
# Cross-Dataset Generalization on ToN-IoT (RQ3)
# Train on CIC-IDS2018, test on ToN-IoT without retraining.
# Feature alignment per Sarhan et al. 2021 (Paper 7 in our lit review).

toniot_path = "/kaggle/input/datasets/dhoogla/nftoniotv2/NF-ToN-IoT-V2.parquet"
df_ton = pd.read_parquet(toniot_path).sample(n=500_000, random_state=42).reset_index(drop=True)
df_ton.columns = df_ton.columns.str.strip()
print(f"ToN-IoT shape: {df_ton.shape}")
print(df_ton['Attack'].value_counts().to_string())

FEATURE_MAP = {
    "FLOW_DURATION_MILLISECONDS": "Flow Duration",
    "IN_PKTS":                    "Total Fwd Packets",
    "OUT_PKTS":                   "Total Backward Packets",
    "IN_BYTES":                   "Fwd Packets Length Total",
    "OUT_BYTES":                  "Bwd Packets Length Total",
    "MAX_IP_PKT_LEN":             "Fwd Packet Length Max",
    "MIN_IP_PKT_LEN":             "Fwd Packet Length Min",
    "LONGEST_FLOW_PKT":           "Packet Length Max",
    "SHORTEST_FLOW_PKT":          "Packet Length Min",
    "SRC_TO_DST_SECOND_BYTES":    "Flow Bytes/s",
    "SRC_TO_DST_AVG_THROUGHPUT":  "Fwd Header Length",
    "DST_TO_SRC_AVG_THROUGHPUT":  "Bwd Header Length",
    "TCP_FLAGS":                  "Fwd PSH Flags",
    "TCP_WIN_MAX_IN":              "Init Fwd Win Bytes",
    "TCP_WIN_MAX_OUT":             "Init Bwd Win Bytes",
    "RETRANSMITTED_IN_PKTS":      "Fwd Avg Packets/Bulk",
    "RETRANSMITTED_OUT_PKTS":     "Bwd Avg Packets/Bulk",
    "RETRANSMITTED_IN_BYTES":     "Fwd Avg Bytes/Bulk",
    "RETRANSMITTED_OUT_BYTES":    "Bwd Avg Bytes/Bulk",
    "NUM_PKTS_UP_TO_128_BYTES":   "Subflow Fwd Packets",
    "NUM_PKTS_1024_TO_1514_BYTES": "Subflow Bwd Packets",
}

cic_features = df.drop(columns=['Label']).columns.tolist()
reverse_map = {v: k for k, v in FEATURE_MAP.items()}
X_ton = np.zeros((len(df_ton), len(cic_features)), dtype=np.float64)
matched, unmatched = [], []
for i, cic_feat in enumerate(cic_features):
    if cic_feat in reverse_map and reverse_map[cic_feat] in df_ton.columns:
        ton_col = reverse_map[cic_feat]
        values = pd.to_numeric(df_ton[ton_col], errors='coerce').fillna(0).values
        if ton_col == "FLOW_DURATION_MILLISECONDS": values = values * 1000
        X_ton[:, i] = values
        matched.append(f"{cic_feat} <- {ton_col}")
    else:
        unmatched.append(cic_feat)
print(f"Matched features: {len(matched)} / {len(cic_features)}")
for m in matched: print(f'  {m}')
print(f"Zero-filled: {len(unmatched)}")

LABEL_MAP = {
    "benign": "Benign", "password": "FTP-BruteForce",
    "dos": "DoS attacks-Hulk", "ddos": "DDoS attacks-LOIC-HTTP",
    "injection": "SQL Injection", "xss": "Brute Force -XSS",
    "backdoor": "Infilteration", "scanning": "Infilteration",
    "ransomware": "Infilteration", "mitm": "Infilteration",
}
df_ton['mapped_label'] = (
    df_ton['Attack'].astype(str).str.lower().str.strip()
    .map(LABEL_MAP).fillna("Unknown-Attack")
)
print(df_ton['mapped_label'].value_counts().to_string())

known_labels = set(label_encoder.classes_)
mask = np.array([l in known_labels for l in df_ton['mapped_label'].values])
X_ton_known = X_ton[mask]
y_ton_known = label_encoder.transform(df_ton['mapped_label'].values[mask])
print(f"Filtered: {len(X_ton_known)}, Dropped: {(~mask).sum()}")

X_ton_known = np.nan_to_num(np.clip(X_ton_known, -1e10, 1e10).astype(np.float32))
X_ton_scaled = np.nan_to_num(scaler.transform(X_ton_known)).astype(np.float32)

sess = ort.InferenceSession("/kaggle/working/threat_mlp_fp32.onnx", providers=["CPUExecutionProvider"])
ton_preds = []
for i in range(0, len(X_ton_scaled), 512):
    out = sess.run(None, {"input": X_ton_scaled[i:i+512]})[0]
    ton_preds.extend(np.argmax(out, axis=1))
ton_preds = np.array(ton_preds)
ton_f1 = f1_score(y_ton_known, ton_preds, average="macro")

print("\nRQ3 CROSS-DATASET REPORT")
print(f"CIC-IDS2018 (in-distribution)   Macro-F1: {fp32_results['macro_f1']:.4f}")
print(f"ToN-IoT (out-of-distribution)   Macro-F1: {ton_f1:.4f}")
print(f"F1 Drop: {fp32_results['macro_f1'] - ton_f1:.4f}")
print(f"Features matched: {len(matched)} / {len(cic_features)}")

unique_labels = sorted(np.unique(np.concatenate([y_ton_known, ton_preds])))
print(classification_report(y_ton_known, ton_preds,
    labels=unique_labels,
    target_names=[label_encoder.classes_[i] for i in unique_labels],
    zero_division=0))

cross_results = {
    "cic_ids2018_f1": fp32_results["macro_f1"],
    "toniot_f1": float(ton_f1),
    "f1_drop": float(fp32_results["macro_f1"] - ton_f1),
    "toniot_samples_evaluated": int(len(X_ton_known)),
    "features_matched": len(matched),
    "feature_mapping_used": FEATURE_MAP,
}
with open("/kaggle/working/cross_dataset_comparison.json", "w") as f:
    json.dump(cross_results, f, indent=2)
print("Cross-dataset results saved")
